In [0]:
# assign the path to a variable
filepath = "/Volumes/teaching/datasets/assignment/emails.csv"

In [0]:
#Viewing the dataset with print function 
print(dbutils.fs.head(filepath))


[Truncated to first 65536 bytes]
"file","message"
"allen-p/_sent_mail/1.","Message-ID: <18782981.1075855378110.JavaMail.evans@thyme>
Date: Mon, 14 May 2001 16:39:00 -0700 (PDT)
From: phillip.allen@enron.com
To: tim.belden@enron.com
Subject: 
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii
Content-Transfer-Encoding: 7bit
X-From: Phillip K Allen
X-To: Tim Belden <Tim Belden/Enron@EnronXGate>
X-cc: 
X-bcc: 
X-Folder: \Phillip_Allen_Jan2002_1\Allen, Phillip K.\'Sent Mail
X-Origin: Allen-P
X-FileName: pallen (Non-Privileged).pst

Here is our forecast

 "
"allen-p/_sent_mail/10.","Message-ID: <15464986.1075855378456.JavaMail.evans@thyme>
Date: Fri, 4 May 2001 13:51:00 -0700 (PDT)
From: phillip.allen@enron.com
To: john.lavorato@enron.com
Subject: Re:
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii
Content-Transfer-Encoding: 7bit
X-From: Phillip K Allen
X-To: John J Lavorato <John J Lavorato/ENRON@enronXgate@ENRON>
X-cc: 
X-bcc: 
X-Folder: \Phillip_Allen_Jan2002_1\All

In [0]:
#View the dataset using splitlines() for better readability.
for line in dbutils.fs.head(filepath).splitlines():
    print(line)

[Truncated to first 65536 bytes]
"file","message"
"allen-p/_sent_mail/1.","Message-ID: <18782981.1075855378110.JavaMail.evans@thyme>
Date: Mon, 14 May 2001 16:39:00 -0700 (PDT)
From: phillip.allen@enron.com
To: tim.belden@enron.com
Subject: 
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii
Content-Transfer-Encoding: 7bit
X-From: Phillip K Allen
X-To: Tim Belden <Tim Belden/Enron@EnronXGate>
X-cc: 
X-bcc: 
X-Folder: \Phillip_Allen_Jan2002_1\Allen, Phillip K.\'Sent Mail
X-Origin: Allen-P
X-FileName: pallen (Non-Privileged).pst

Here is our forecast

 "
"allen-p/_sent_mail/10.","Message-ID: <15464986.1075855378456.JavaMail.evans@thyme>
Date: Fri, 4 May 2001 13:51:00 -0700 (PDT)
From: phillip.allen@enron.com
To: john.lavorato@enron.com
Subject: Re:
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii
Content-Transfer-Encoding: 7bit
X-From: Phillip K Allen
X-To: John J Lavorato <John J Lavorato/ENRON@enronXgate@ENRON>
X-cc: 
X-bcc: 
X-Folder: \Phillip_Allen_Jan2002_1\All

In [0]:
#Load the dataset

from pyspark.sql import functions as F

df = (
    spark.read
    .option("header", "true")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .csv(filepath)
)

In [0]:
df.count()

517401

In [0]:
#View a sample of the email

sample_emails = df.select("message").limit(10).collect()

for i, row in enumerate(sample_emails, start=1):
    print("=" * 60)
    print(f"Email #{i}")
    print("=" * 60)
    print(row["message"])
    print("\n\n")

Email #1
Message-ID: <18782981.1075855378110.JavaMail.evans@thyme>
Date: Mon, 14 May 2001 16:39:00 -0700 (PDT)
From: phillip.allen@enron.com
To: tim.belden@enron.com
Subject: 
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii
Content-Transfer-Encoding: 7bit
X-From: Phillip K Allen
X-To: Tim Belden <Tim Belden/Enron@EnronXGate>
X-cc: 
X-bcc: 
X-Folder: \Phillip_Allen_Jan2002_1\Allen, Phillip K.\'Sent Mail
X-Origin: Allen-P
X-FileName: pallen (Non-Privileged).pst

Here is our forecast

 



Email #2
Message-ID: <15464986.1075855378456.JavaMail.evans@thyme>
Date: Fri, 4 May 2001 13:51:00 -0700 (PDT)
From: phillip.allen@enron.com
To: john.lavorato@enron.com
Subject: Re:
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii
Content-Transfer-Encoding: 7bit
X-From: Phillip K Allen
X-To: John J Lavorato <John J Lavorato/ENRON@enronXgate@ENRON>
X-cc: 
X-bcc: 
X-Folder: \Phillip_Allen_Jan2002_1\Allen, Phillip K.\'Sent Mail
X-Origin: Allen-P
X-FileName: pallen (Non-Privileged).p

In [0]:
#Check the schema. The 'message' column is very wide and that is why it is showing all the dashes (see the next cell to understand what it is trying to do )

df.printSchema()
df.show(5, truncate=False)

root
 |-- file: string (nullable = true)
 |-- message: string (nullable = true)

+------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
#Split the first column into multiple columns

#DO WE NEED TO CONSIDER CC AND BCC? - ASK ON WEDNESDAY

emails = df \
    .withColumn(
        "Date",
        F.regexp_extract("message", r"Date:\s*(.*)", 1)
    ) \
    .withColumn(
        "From",
        F.regexp_extract("message", r"From:\s*(.*)", 1)
    ) \
    .withColumn(
        "To",
        F.regexp_replace(
            F.regexp_extract("message", r"To:\s*(.*)", 1),
            ",\\s*$", ""   # remove trailing comma + optional spaces
        )
    ) \
    .withColumn(
        "cc",
        F.regexp_extract("message", r"cc:\s*(.*)", 1)
    ) \
    .withColumn(
        "bcc",
        F.regexp_extract("message", r"bcc:\s*(.*)", 1)
    ) \
    .withColumn(
        "Subject",
        F.regexp_extract("message", r"Subject:\s*(.*)", 1)
    )

emails.select("Date", "From", "To", "Subject").show(5, truncate=False)

+-------------------------------------+-----------------------+-----------------------+-----------------+
|Date                                 |From                   |To                     |Subject          |
+-------------------------------------+-----------------------+-----------------------+-----------------+
|Mon, 14 May 2001 16:39:00 -0700 (PDT)|phillip.allen@enron.com|tim.belden@enron.com   |Mime-Version: 1.0|
|Fri, 4 May 2001 13:51:00 -0700 (PDT) |phillip.allen@enron.com|john.lavorato@enron.com|Re:              |
|Wed, 18 Oct 2000 03:00:00 -0700 (PDT)|phillip.allen@enron.com|leah.arsdall@enron.com |Re: test         |
|Mon, 23 Oct 2000 06:13:00 -0700 (PDT)|phillip.allen@enron.com|randall.gay@enron.com  |Mime-Version: 1.0|
|Thu, 31 Aug 2000 05:07:00 -0700 (PDT)|phillip.allen@enron.com|greg.piper@enron.com   |Re: Hello        |
+-------------------------------------+-----------------------+-----------------------+-----------------+
only showing top 5 rows


In [0]:
#Clean up the columns and remove whitespace

emails = (
    emails
    .withColumn("From", F.lower(F.trim(F.col("From"))))
    .withColumn("To", F.lower(F.trim(F.col("To"))))
    .withColumn("Subject", F.trim(F.col("Subject")))
    )        


In [0]:
#Convert Date to Timestamp - This might not be necessary, but it's easier to work with Timestamps
emails = emails.withColumn(
    "Date_ts",
    F.to_timestamp("Date", "EEE, d MMM yyyy HH:mm:ss Z")
)

In [0]:
# Split mutiple recipients in the two column into an array, exploded the array so each recipient is in a separate row, trim white space .
emails_recipients = (
    emails
    .withColumn("To_array", F.split(F.col("To"), ","))
    .withColumn("Recipient", F.explode(F.col("To_array")))     
    .withColumn("Recipient", F.trim(F.col("Recipient")))
)
  

In [0]:
# show columns
emails_recipients.select("To", "Recipient","To_array").show(5, truncate=False)
emails_recipients.printSchema()

+-----------------------+-----------------------+-------------------------+
|To                     |Recipient              |To_array                 |
+-----------------------+-----------------------+-------------------------+
|tim.belden@enron.com   |tim.belden@enron.com   |[tim.belden@enron.com]   |
|john.lavorato@enron.com|john.lavorato@enron.com|[john.lavorato@enron.com]|
|leah.arsdall@enron.com |leah.arsdall@enron.com |[leah.arsdall@enron.com] |
|randall.gay@enron.com  |randall.gay@enron.com  |[randall.gay@enron.com]  |
|greg.piper@enron.com   |greg.piper@enron.com   |[greg.piper@enron.com]   |
+-----------------------+-----------------------+-------------------------+
only showing top 5 rows
root
 |-- file: string (nullable = true)
 |-- message: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- From: string (nullable = true)
 |-- To: string (nullable = true)
 |-- cc: string (nullable = true)
 |-- bcc: string (nullable = true)
 |-- Subject: string (nullable = true

In [0]:
#this cells shows the emails with multiple recipients

emails_recipients.filter(F.size(F.col("To_array")) >= 2).select("To", "Recipient","To_array").show(10,truncate=False)

+------------------------------------------------------------------------+-----------------------------+----------------------------------------------------------------------------+
|To                                                                      |Recipient                    |To_array                                                                    |
+------------------------------------------------------------------------+-----------------------------+----------------------------------------------------------------------------+
|david.l.johnson@enron.com, john.shafer@enron.com                        |david.l.johnson@enron.com    |[david.l.johnson@enron.com,  john.shafer@enron.com]                         |
|david.l.johnson@enron.com, john.shafer@enron.com                        |john.shafer@enron.com        |[david.l.johnson@enron.com,  john.shafer@enron.com]                         |
|paul.lucci@enron.com, kenneth.shulklapper@enron.com                     |paul.lucci@enron

In [0]:
#check the new schema to see the changes we've made
emails.printSchema()

root
 |-- file: string (nullable = true)
 |-- message: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- From: string (nullable = true)
 |-- To: string (nullable = true)
 |-- cc: string (nullable = true)
 |-- bcc: string (nullable = true)
 |-- Subject: string (nullable = true)
 |-- Date_ts: timestamp (nullable = true)



In [0]:
#Creating a dataframe that only contains the From column
dmainDF = emails.select("From")

In [0]:
dmainDF.show(10)

+--------------------+
|                From|
+--------------------+
|phillip.allen@enr...|
|phillip.allen@enr...|
|phillip.allen@enr...|
|phillip.allen@enr...|
|phillip.allen@enr...|
|phillip.allen@enr...|
|phillip.allen@enr...|
|phillip.allen@enr...|
|phillip.allen@enr...|
|phillip.allen@enr...|
+--------------------+
only showing top 10 rows


In [0]:
#Modifying the values of each of the columns to only show the part after the '@'
from pyspark.sql.functions import substring_index

domain2DF = dmainDF.withColumn("Domain", substring_index("From", "@", -1)) 
display(domain2DF)


#assisted by Genie code

From,Domain
phillip.allen@enron.com,enron.com
phillip.allen@enron.com,enron.com
phillip.allen@enron.com,enron.com
phillip.allen@enron.com,enron.com
phillip.allen@enron.com,enron.com
phillip.allen@enron.com,enron.com
phillip.allen@enron.com,enron.com
phillip.allen@enron.com,enron.com
phillip.allen@enron.com,enron.com
phillip.allen@enron.com,enron.com


In [0]:
#Ceating dataframe with only the Domain column
domain3df = domain2DF.select("Domain")
display(domain3df)

Domain
enron.com
enron.com
enron.com
enron.com
enron.com
enron.com
enron.com
enron.com
enron.com
enron.com


In [0]:
#Grouping all similar entries in the Domain column, counting each appearance and recording it in the count column
grouped_count_df = (
    domain3df
    .groupBy("Domain")
    .count()
    .orderBy("count", ascending=False)
)

display(grouped_count_df)

#assisted by week_8 workshop

Domain,count
enron.com,426229
aol.com,2803
hotmail.com,2427
mailman.enron.com,1775
txu.com,1653
enron.com>,1556
nymex.com,1438
haas.berkeley.edu,1317
yahoo.com,1309
carrfut.com,1303


In [0]:
#Listing the top 10 domains excluding any that start with 'enron'
Top_10DF = grouped_count_df.filter(~F.col("domain").startswith("enron"))
Top_10DF.show(10, truncate=False)

#assisted by Genie code and week_4 workshop

+---------------------------+-----+
|Domain                     |count|
+---------------------------+-----+
|aol.com                    |2803 |
|hotmail.com                |2427 |
|mailman.enron.com          |1775 |
|txu.com                    |1653 |
|nymex.com                  |1438 |
|haas.berkeley.edu          |1317 |
|yahoo.com                  |1309 |
|carrfut.com                |1303 |
|ccomad3.uu.commissioner.com|877  |
|caiso.com                  |838  |
+---------------------------+-----+
only showing top 10 rows
